In [1]:
import igraph
from igraph import Graph, EdgeSeq, plot

import plotly
import plotly.graph_objects as go

import time
import random
import math

In [34]:
# Sets off the script by determining the number of variables and the number of instance creations
# trackable: keeps track of variable count per generation
# iterable: keeps track of instances created based on the number of variables. Is passed to VariableDetermination
# varCount: a nested array showing the eponentiated parent\child relationships
def Main():
    iterable = [1]
    trackable = [] 
    varCount = []
    for i in range(5): # Represents the number of runs | events being simulated
        if(iterable[i] > 0):
            varCount.append(VariableDetermination(iterable[i]))#varCount is a multi-dimensional list, where each row represents a different generation
            if(len(varCount[i]) > 1):
                iSum = 0
                tSum = 0
                for _ in varCount[i]:
                    determinedValue = Determination(_) 
                    tSum += _
                    iSum += determinedValue
                    
                trackable.append(tSum)
                iterable.append(iSum)
            else:
                temp = varCount[i][0]
                determinedValue = Determination(temp) 
                trackable.append(temp)
                iterable.append(determinedValue)
        else:
            break;
    CreateTree(varCount, trackable)

# Sets the variable count for each parent value passed through
def VariableDetermination(instanceCount):
    random.seed(time.time())
    return [random.randint(0,3) for _ in range(instanceCount)]
    
# Except for 0, any integer passed through gets exponentiated. If 0 is passed through, then 0 is returned. This ensures that no new instances are created after an instance death
def Determination(ele):
    if(ele == 0):
        return ele
    else:
        return 2**ele

In [32]:
# Obtains the tree structure that is used to build the tree graph then prints it to SVG format
# treeStruct: a multidimensional list consisting of all the edges\relationships used to build the tree
# treeVertices: a flattened list based on treeStruct that ensures that all vertices are created in the tree
def CreateTree(varCount, trackable):
    label = trackable
    treeStructure = SetEdges(varCount)
    treeVertices = list(dict.fromkeys([i for _ in treeStructure for i in _]))
    # print(f"Vertex Names: {treeVertices}")
    graph = Graph(directed = True)
    graph.add_vertices(treeVertices)
    graph.add_edges(treeStructure)
    layout = graph.layout_reingold_tilford(root=[0])
    imageSave = Incrementor()
    plot(
        graph,
        layout = layout,
        vertex_label_size = 8,
        vertex_size = 10,
        vertex_color ="blue",
        # vertex_label = gra
        bbox = (2000,1200),
        margin = 50,
        target = imageSave
    )

# Goes through a three step process
# unique_varCount: hex names are used to uniquely identify each element
# struct: a dictionary thats formatted in {parent: [children]} format. Is created from the SetEdges function
# edges: the [parent, child] version of the struct variable
def SetEdges(varCount):
    unique_varCount = UniqueVertices(varCount) 
    struct = StructuredAssociations(varCount, unique_varCount)
    edges = CreateEdges(struct)
    return edges

In [33]:
# Iterates through sequential integers and converts them to their hex equivalent. These hex values are used to uniquely identify each element in the nested list    
def UniqueVertices(varCount):
    incremental = 0
    tempOuter = []
    tempVertices = []
    for group in varCount:
        tempInner = []
        for item in group:
            for i in range(item):
                incremental += 1
                tempInner.append(hex(incremental))
        tempOuter.append(tempInner)
    return tempOuter


# Uses the poweredCount variables and unique_varCount variables to:
# 1. Determine which elements receive which unique hex name
# 2. Determine which child element is to be paired with each parent

# tempStruct: the disctionary holding {parent: [children]} pairs
# oIndex: identifies the index, in poweredCount, which will be used as the source of children
# parentIndex: holds the index of the parent subset within poweredCount
# unique_skipper: instances with 0 children do not have a relationship with hex names occuring within the next generation. Thee skipper variable is meant to ensure that the graph adheres to this rule

# These two work together to assign hex values to the child set
# subStart: indicates the start index for which the hex name of the children associated with a particular parent
# subEnd: indicated the end value +1, for the subset

# dupCheck: a dictionary consisting of {parent value: [indices]} form
# orderedSet: consists of a list of predetermined steps for the subStart and subEnd variables

# uniqueParent: contains the hex name of the parent in the {parent: [children]} pair
# uniqueChild: contains the hex name of all associated children
def StructuredAssociations(poweredCount, unique_varCount):
    tempStruct = {} 
    for outer in poweredCount: 
        oIndex = poweredCount.index(outer) 
        parentIndex = 0
        if(oIndex == 0):
                continue
        else:
            parentIndex = oIndex-1
            parentSet = poweredCount[parentIndex]
            parentCount = len(poweredCount[parentIndex])
            subStart = 0
            subEnd = 0
            dupCheck = DuplicateCounter(parentSet)
            orderedSet = OrderedSubSets(parentSet, dupCheck)
            unique_skipper = FindSkips(dupCheck, orderedSet)
            for counter in orderedSet:
                cIndex = orderedSet.index(counter)
                if(counter == subEnd):
                    continue
                else:
                    subEnd = counter
                    if(cIndex in unique_skipper):
                        continue
                    else:
                        uniqueParent = unique_varCount[parentIndex][cIndex]
                        uniqueChild = unique_varCount[oIndex][subStart:subEnd]
                        if(uniqueParent in tempStruct.keys()):
                            continue
                        else:
                            tempStruct.update({uniqueParent: uniqueChild})
                        subStart = subEnd
    # print(f"Structure: {tempStruct}")
    return tempStruct

# the tree structure dictionary is transformed into a nested list of [parent, child] pairs
def CreateEdges(struct):
    edges = []
    for key in struct.keys():
        for node in struct[key]:
            edges.append([key, node])
    # print(f"Edges: {edges}")
    return edges            
           

# This function takes on two important tasks: 
# 1. Sends the parent set through duplicate identification (DuplicateCounter) function
# 2. Determines the length of child subsets based on the indices that each parent element occurred in and stores them in a list that is returned to the calling function
def OrderedSubSets(parent, dupCheck):
    orderedList = []
    tempSum = 0
    for i in range(len(parent)):
        for value in dupCheck.keys():
            if(i in dupCheck[value]):
                tempSum += Determination(value)
                orderedList.append(tempSum)
    return list(orderedList)

# Looks for duplicate values in the parent list -- duplicates cause confusion regarding the start and end indices which are used in determining the child subset to be paired with the parent set
# This function creates a dictionary where the keys = the value of each item in the parent set; the values = the indices in which each key occurs
def DuplicateCounter(parent):
    dupCheck = {}
    for child in parent:
        indices = [i for i, x in enumerate(parent) if x == child]
        dupCheck[child] = indices    
    return dupCheck

# The predetermined values in orderedSet are used to identify parents with 0 children. It works due to the fact that parents without children will contain the same value as the index before it
def FindSkips(dupCheck, orderedSet):
    temp = []
    for x in orderedSet:
        xIndex = orderedSet.index(x)
        if(xIndex != len(orderedSet)-1):
            nextX = xIndex + 1
            if(orderedSet[nextX] == orderedSet[xIndex]):
                temp.append(nextX)
    return temp

In [35]:
# Used for saving the resulting graph
# Will need to initialize increment 

#increment = 0
def Incrementor():
    global increment
    increment += 1
    imageSave = f".\\Graph Attempts\\Visualization No {increment}.svg"
    return imageSave
    
Main()